# Full run vs resume in Metaflow

Comparing a fresh run against a resumed run during iterative model development. The goal is to understand how `resume` reuses past steps and where it picks up, so you don't re-execute expensive data-loading or preprocessing steps while iterating on training logic.

In [ ]:
from metaflow import FlowSpec, step, Parameter, current
import numpy as np
from time import sleep

## The flow

Three stages mimic an iterative ML workflow:
- `load_data` — simulates an expensive download (sleep 2s)
- `train` — trains a dummy classifier, configurable via `--model` parameter
- `evaluate` — reports accuracy

After a full run, I'll resume with a different model type to see which steps get replayed.

In [ ]:
class IterativeMLFlow(FlowSpec):

    model = Parameter("model", help="Model type to use", default="logistic")

    @step
    def start(self):
        print("Starting flow")
        self.next(self.load_data)

    @step
    def load_data(self):
        """Simulate an expensive data-load step."""
        print("Loading data... (simulated 2s delay)")
        sleep(2)
        self.X = np.random.randn(500, 10)
        self.y = (self.X[:, 0] + self.X[:, 1] > 0).astype(int)
        print(f"Data loaded: X shape {self.X.shape}")
        self.next(self.train)

    @step
    def train(self):
        print(f"Training with model={self.model}...")
        # Dummy training — real projects would call sklearn / torch
        sleep(1)
        from sklearn.linear_model import LogisticRegression
        from sklearn.ensemble import RandomForestClassifier
        if self.model == "random_forest":
            clf = RandomForestClassifier(n_estimators=10)
        else:
            clf = LogisticRegression(max_iter=100)
        clf.fit(self.X, self.y)
        self.model_obj = clf
        print("Training complete")
        self.next(self.evaluate)

    @step
    def evaluate(self):
        acc = self.model_obj.score(self.X, self.y)
        self.accuracy = acc
        print(f"Accuracy: {acc:.3f}")
        self.next(self.end)

    @step
    def end(self):
        print(f"Flow complete. Best accuracy: {self.accuracy:.3f}")

## Full run

Run from scratch with the default `logistic` model. All four steps execute.

In [ ]:
if __name__ == "__main__":
    IterativeMLFlow()

```bash
# CLI equivalent
python 2026-06-17-full-run-vs-resume.ipynb run
```

Expected output:
```
Starting flow
Loading data... (simulated 2s delay)
Data loaded: X shape (500, 10)
Training with model=logistic...
Training complete
Accuracy: 0.876
Flow complete. Best accuracy: 0.876
```

## Resume with a different model

Now resume from the `train` step using `--model random_forest`. Metaflow skips `load_data` and reuses its artifacts, then re-executes `train` and `evaluate`.

```bash
python 2026-06-17-full-run-vs-resume.ipynb resume train --model random_forest
```

Expected output (notice no "Loading data..." line):
```
Resuming from step train
Training with model=random_forest...
Training complete
Accuracy: 0.942
Flow complete. Best accuracy: 0.942
```

## What the comparison tells us

- `resume` skips every step before the target — `load_data` is not re-executed, which saves the 2s delay and preserves the same data split. This is critical for reproducibility during iteration: you change only the training code while keeping data fixed.
- The resumed run gets a new run ID but inherits the artifact store from the original upstream steps. The data (`self.X`, `self.y`) comes from the original run's cache.
- You can resume from any step, not just the one right after data loading. Useful when the expensive step is feature engineering or a hyperparameter sweep.

One thing to watch: if the data-loading logic changes (e.g., you edit the `load_data` step code), `resume` still uses the cached artifacts from the original run — not the new code. To force re-execution of a cached step, delete the run or use `--run-id` to point at a different base run.

## Verify it works

After running both commands, list the runs:

```bash
python 2026-06-17-full-run-vs-resume.ipynb list
```

You should see two runs — the original full run and the shorter resumed run. The resumed run's path to artifacts shows it depends on the original run's `load_data`.

## Common gotchas

- Metaflow's `resume` is tied to the **latest** run of the same flow by default. Use `--run-id` to pick a specific base run.
- If you delete the original run's data in the metadata service, the resumed run can't inherit artifacts and will fail with a `DataArtifactNotFound` error.
- The `@conda` or `@batch` decorators on resumed steps still apply — make sure the environment matches the original run's upstream steps.